In [1]:
%matplotlib notebook

In [2]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

import jax
import jax.numpy as jnp
import numpy as onp
from tqdm import tqdm
import matplotlib.pyplot as plt

from msmjax.kernels import split_one_over_r_kernel, SofteningFunctionOneOverR
from msmjax.shortrange import make_compute_U_zero_with_neighborlist, make_compute_f_zero_with_neighborlist, make_compute_U_and_f_zero_with_neighborlist
from msmjax.gridops_multidim import set_up_grids_all_levels
from msmjax.gridops_multidim import create_compute_U_oneplus, create_compute_U_and_f_oneplus


In [177]:
AVG_NEIGHBOR_DISTANCE = 2.5
N_DIM = 3
PBCS = [False] * N_DIM

N_LEVELS = 4
ALPHA = 3
P = 6
MU = 3
LEVEL_ONE_GRIDSPACING = AVG_NEIGHBOR_DISTANCE
LEVEL_ZERO_CUTOFF = ALPHA * LEVEL_ONE_GRIDSPACING

In [178]:
kernels = split_one_over_r_kernel(
        max_level=N_LEVELS,
        level_zero_cutoff=LEVEL_ZERO_CUTOFF,
        softening_function=SofteningFunctionOneOverR(P),
    )

In [202]:
rng = onp.random.default_rng(2489)

# Randomly drawn particle positions
n_particles = 5000
side_length = n_particles ** (1. / N_DIM) * AVG_NEIGHBOR_DISTANCE
box_lengths = jnp.array([side_length] * N_DIM)
pos = rng.uniform(low=onp.zeros_like(box_lengths), high=box_lengths, size=(n_particles, N_DIM))

# # Particles placed on a regular grid
# n_points_per_direction = 9
# n_particles = n_points_per_direction**N_DIM
# side_length = n_points_per_direction * AVG_NEIGHBOR_DISTANCE
# box_lengths = jnp.array([side_length] * N_DIM)
# mg = onp.meshgrid(*([onp.arange(n_points_per_direction)] * N_DIM), indexing="ij")
# pos = onp.stack([m.ravel() for m in mg], axis=1) * AVG_NEIGHBOR_DISTANCE

chg = rng.uniform(low=-1.0, high=1.0, size=n_particles)

pos = jnp.array(pos)
chg = jnp.array(chg)

In [203]:
dummy_box_lengths_nonperiodic_shortrange = jnp.ones(N_DIM)

neighbor_fun, compute_U_zero_with_neighborlist = make_compute_U_zero_with_neighborlist(
    kernels=kernels,
    cutoff=LEVEL_ZERO_CUTOFF,
    box_lengths=dummy_box_lengths_nonperiodic_shortrange,
    pbcs=PBCS,
)
nbl_allocate_fun = neighbor_fun.allocate
nbl_update_fun = neighbor_fun.update
neighbor_list = nbl_allocate_fun(pos)

@jax.jit
def wrapped_compute_U_zero_with_neighborlist(positions, charges):
    updated_neighbor_list = neighbor_fun.update(positions, neighbor_list)
    return compute_U_zero_with_neighborlist(positions, charges, updated_neighbor_list.idx)

/home/florian/anaconda3/envs/msmjax-dev/lib/python3.8/site-packages/jax/_src/ops/scatter.py:92: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=int64 to dtype=int32. In future JAX releases this will result in an error.
  warnings.warn("scatter inputs have incompatible types: cannot safely cast "


In [231]:
def make_compute_U_zero_reference(kernels, periodic: bool):
    k_0 = kernels[0]
    sum_of_higher_kernels_at_zero = jnp.sum(
        jnp.asarray([k(0.0) for k in kernels[1:]])
    )
    
    def compute_U_zero_reference(positions, charges):
        R_ij = positions[:, jnp.newaxis, :] - positions
        if periodic:
            R_ij -= jnp.rint(R_ij)
        qi_qj = charges[:, jnp.newaxis] * charges
        indices_triu = jnp.triu_indices(positions.shape[0], k=1)
        r_ij_triu = jnp.linalg.norm(R_ij[indices_triu], axis=1)
        qi_qj_triu = qi_qj[indices_triu]
        
        pair_term = (qi_qj_triu * jax.vmap(k_0)(r_ij_triu)).sum()
        self_energy_term = 0.5 * jnp.diag(qi_qj).sum() * sum_of_higher_kernels_at_zero
    
        return pair_term - self_energy_term
    
    return compute_U_zero_reference

compute_U_zero_reference = jax.jit(make_compute_U_zero_reference(kernels, periodic=PBCS[0]))

In [232]:
U_zero_nbl = wrapped_compute_U_zero_with_neighborlist(pos, chg)
U_zero_reference = compute_U_zero_reference(pos, chg)

assert jnp.isclose(U_zero_nbl, U_zero_reference)

print(U_zero_reference)

-301.66089831736684


In [217]:
jax.device_put(pos)
jax.device_put(chg)

%timeit compute_U_zero_reference(pos, chg).block_until_ready()

16.9 ms ± 677 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [235]:
jax.device_put(pos)
jax.device_put(chg)

%timeit compute_U_zero_reference(pos, chg).block_until_ready()

16.6 ms ± 505 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [236]:
jax.device_put(pos)
jax.device_put(chg)

%timeit wrapped_compute_U_zero_with_neighborlist(pos, chg).block_until_ready()

671 µs ± 6.52 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [182]:
import matplotlib.pyplot as plt

try:
    x, y, z, = pos[:, 0], pos[:, 1], pos[:, 2]

    fig = plt.figure()
    ax = fig.add_subplot(projection="3d")
    ax.scatter(x.ravel(), y.ravel(), z.ravel(), c=chg, s=10)

    plt.show()
except ValueError:
    pass

<IPython.core.display.Javascript object>